# [17.1] Checkpoint Archaeology and Mechanism Emergence

> **Local-first extension.** This section teaches checkpoint archaeology as a bounded mechanism-emergence workflow. The solved notebook uses deterministic toy checkpoint trajectories and a CPU-feasible live mod-13 model organism; the CUDA report reruns the same train/save/reload/control path on the RTX 5090.

## Core Question

When can we say a mechanism emerged during training?

A good answer is not just a rising metric. It needs a threshold, a stable run above the threshold, a phase-transition or timing report, saved checkpoints that can be reloaded, and a negative control that does not look like the same mechanism.

## Learning Objectives

By the end of this notebook you should be able to separate first crossing from stable crossing, count monotonicity violations, detect adjacent jumps, reject overstrong random controls, compare toy developmental timings without ranking the control, and read a CUDA checkpoint report without widening the finite-model-organism claim.


In [1]:
from __future__ import annotations

import json
import sys
import tempfile
from pathlib import Path

import torch as t

GT_TIER = "GT-0"
EXERCISE_ID = "17_1_checkpoint_archaeology_and_mechanism_emergence"
DIFFICULTY = 4
IMPORTANCE = 3
EXPECTED_RUNTIME = "seconds on toy contract; minutes on local real-model path"
REQUIRES_GPU = True

chapter = "chapter17_training_dynamics"
section = "part1_checkpoint_archaeology"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_checkpoint_archaeology.tests as tests
import part1_checkpoint_archaeology.utils as utils

from arena_ext.training_dynamics import (
    DevelopmentalComparisonReport,
    MechanismEmergenceReport,
    PhaseTransitionReport,
    RandomControlReport,
    toy_training_trajectories,
)


## Exercise 1 - Checkpoint Series Helpers

The smallest useful checkpoint-archaeology primitive is a validated series and two different crossing notions: first crossing and stable crossing.

> Difficulty: medium  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_first_threshold_crossing_finds_first_crossing_and_validates_inputs` passed!
All tests in `test_stable_threshold_step_requires_consecutive_checkpoints` passed!
```

</details>

<details>
<summary>Help - how do first and stable crossings differ?</summary>

A first crossing can be a single lucky spike. A stable crossing starts a window of `min_consecutive` checkpoints that all stay above threshold.

</details>

<details>
<summary>Common bugs</summary>

- Returning the tensor index rather than the checkpoint step.
- Sorting non-monotone checkpoints instead of rejecting them.
- Using strict `>` and missing a value exactly equal to threshold.

</details>

<details>
<summary>Solution</summary>

Validate shape, monotone step order, and finite values. Use `values >= threshold`, then return the corresponding checkpoint step.

</details>


In [2]:
def _validate_checkpoint_series(
    steps: t.Tensor,
    values: t.Tensor,
) -> tuple[t.Tensor, t.Tensor]:
    steps = t.as_tensor(steps).flatten().long()
    values = t.as_tensor(values).flatten().float()
    if steps.numel() == 0:
        raise ValueError("checkpoint series must be nonempty.")
    if steps.numel() != values.numel():
        raise ValueError("steps and values must have the same length.")
    if steps.numel() > 1 and not bool((steps[1:] > steps[:-1]).all().item()):
        raise ValueError("checkpoint steps must be strictly increasing.")
    if not bool(t.isfinite(values).all().item()):
        raise ValueError("checkpoint values must be finite.")
    return steps, values


def first_threshold_crossing(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    threshold: float,
) -> int | None:
    steps, values = _validate_checkpoint_series(steps, values)
    crossed = values >= threshold
    if not bool(crossed.any().item()):
        return None
    index = int(t.nonzero(crossed, as_tuple=False)[0].item())
    return int(steps[index].item())


def stable_threshold_step(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    threshold: float,
    min_consecutive: int = 2,
) -> int | None:
    if min_consecutive <= 0:
        raise ValueError("min_consecutive must be positive.")
    steps, values = _validate_checkpoint_series(steps, values)
    if values.numel() < min_consecutive:
        return None
    for index in range(values.numel() - min_consecutive + 1):
        window = values[index : index + min_consecutive]
        if bool((window >= threshold).all().item()):
            return int(steps[index].item())
    return None


tests.test_first_threshold_crossing_finds_first_crossing_and_validates_inputs(
    first_threshold_crossing
)
tests.test_stable_threshold_step_requires_consecutive_checkpoints(stable_threshold_step)


All tests in `test_first_threshold_crossing_finds_first_crossing_and_validates_inputs` passed!
All tests in `test_stable_threshold_step_requires_consecutive_checkpoints` passed!


## Exercise 2 - Emergence Reports

A report should make the claim auditable. Keep the threshold, first crossing, stable crossing, peak checkpoint, peak value, monotonicity violations, and final boolean.

> Difficulty: medium  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_mechanism_emergence_report_tracks_peak_and_monotonicity` passed!
```

</details>

<details>
<summary>Help - what should the report make visible?</summary>

A reviewer should be able to tell whether the mechanism is stable, whether the peak came later, and whether the metric regressed between checkpoints.

</details>

<details>
<summary>Common bugs</summary>

- Reporting only first crossing and hiding stable crossing.
- Letting NaNs pass as evidence.
- Counting tolerated numerical noise as a real monotonicity failure.

</details>

<details>
<summary>Solution</summary>

Count adjacent decreases larger than tolerance, reuse the crossing helpers, compute the peak with `argmax`, and require stable crossing plus limited monotonicity violations.

</details>


In [3]:
def monotonicity_violations(values: t.Tensor, *, tolerance: float = 0.0) -> int:
    values = t.as_tensor(values).flatten().float()
    if values.numel() == 0:
        raise ValueError("values must be nonempty.")
    if not bool(t.isfinite(values).all().item()):
        raise ValueError("values must be finite.")
    if values.numel() == 1:
        return 0
    decreases = values[:-1] - values[1:]
    return int((decreases > tolerance).sum().item())


def mechanism_emergence_report(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    metric_name: str = "mechanism_metric",
    threshold: float = 0.6,
    min_consecutive: int = 2,
    max_monotonicity_violations: int = 1,
) -> MechanismEmergenceReport:
    steps, values = _validate_checkpoint_series(steps, values)
    first_crossing = first_threshold_crossing(steps, values, threshold=threshold)
    stable_from = stable_threshold_step(
        steps,
        values,
        threshold=threshold,
        min_consecutive=min_consecutive,
    )
    peak_index = int(t.argmax(values).item())
    violations = monotonicity_violations(values)
    return MechanismEmergenceReport(
        metric_name=metric_name,
        threshold=threshold,
        first_crossing_step=first_crossing,
        stable_from_step=stable_from,
        peak_step=int(steps[peak_index].item()),
        peak_value=float(values[peak_index].item()),
        monotonicity_violations=violations,
        emerged=stable_from is not None and violations <= max_monotonicity_violations,
    )


tests.test_mechanism_emergence_report_tracks_peak_and_monotonicity(
    mechanism_emergence_report
)


All tests in `test_mechanism_emergence_report_tracks_peak_and_monotonicity` passed!


## Exercise 3 - Phase Transitions and Controls

The largest adjacent jump is a developmental warning, not proof. The random-control report is the falsification check: if the control trajectory also looks strong, the mechanism story should fail.

> Difficulty: medium  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_phase_transition_report_detects_largest_adjacent_jump` passed!
All tests in `test_random_control_report_rejects_overstrong_control` passed!
```

</details>

<details>
<summary>Help - why adjacent jumps?</summary>

A phase-transition warning asks where the metric changed fastest between neighboring checkpoints. Comparing every checkpoint to step zero answers a different question.

</details>

<details>
<summary>Common bugs</summary>

- Returning the pre-jump checkpoint instead of the post-jump checkpoint.
- Checking only the final random-control value instead of its peak.
- Letting a strong control pass because the target run also passed.

</details>

<details>
<summary>Solution</summary>

Take `values[1:] - values[:-1]`, report the post-jump step, and gate the random control on its maximum value.

</details>


In [4]:
def phase_transition_report(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    metric_name: str = "mechanism_metric",
    min_jump: float = 0.25,
) -> PhaseTransitionReport:
    steps, values = _validate_checkpoint_series(steps, values)
    if values.numel() < 2:
        raise ValueError("at least two checkpoints are required.")
    jumps = values[1:] - values[:-1]
    transition_index = int(t.argmax(jumps).item())
    jump = float(jumps[transition_index].item())
    return PhaseTransitionReport(
        metric_name=metric_name,
        transition_step=int(steps[transition_index + 1].item()),
        pre_value=float(values[transition_index].item()),
        post_value=float(values[transition_index + 1].item()),
        jump=jump,
        phase_transition_detected=jump >= min_jump,
    )


def random_control_report(
    values: t.Tensor,
    *,
    metric_name: str = "random_control",
    max_allowed_value: float = 0.3,
) -> RandomControlReport:
    values = t.as_tensor(values).flatten().float()
    if values.numel() == 0:
        raise ValueError("control values must be nonempty.")
    if not bool(t.isfinite(values).all().item()):
        raise ValueError("control values must be finite.")
    peak_value = float(values.max().item())
    return RandomControlReport(
        metric_name=metric_name,
        peak_value=peak_value,
        max_allowed_value=max_allowed_value,
        control_passed=peak_value <= max_allowed_value,
    )


tests.test_phase_transition_report_detects_largest_adjacent_jump(
    phase_transition_report
)
tests.test_random_control_report_rejects_overstrong_control(random_control_report)


All tests in `test_phase_transition_report_detects_largest_adjacent_jump` passed!
All tests in `test_random_control_report_rejects_overstrong_control` passed!


## Exercise 4 - Developmental Comparisons

The toy trajectories include AR, JEPA, diffusion, Mamba-style curves, and a random control. The control should be visible in the report but excluded from earliest/latest model-family ordering.

> Difficulty: medium  
> Importance: medium

<details>
<summary>Expected output</summary>

```text
All tests in `test_developmental_comparison_excludes_random_control_from_ordering` passed!
```

</details>

<details>
<summary>Help - why not rank the random control?</summary>

The control is not a model family whose timing you want to compare. It is a falsification check for whether the family timing story is meaningful.

</details>

<details>
<summary>Common bugs</summary>

- Dropping the random control from the report entirely.
- Treating a missing stable step as zero.
- Including the random control in earliest/latest ordering.

</details>

<details>
<summary>Solution</summary>

Compute stable steps for every trajectory, filter the control out of ordering, then separately report whether the control stayed below threshold.

</details>


In [5]:
def developmental_comparison_report(
    steps: t.Tensor,
    family_values: dict[str, t.Tensor],
    *,
    threshold: float = 0.6,
    min_consecutive: int = 2,
    control_name: str = "random_control",
) -> DevelopmentalComparisonReport:
    if not family_values:
        raise ValueError("family_values must contain at least one trajectory.")
    emergence_steps: dict[str, int | None] = {}
    for family, values in family_values.items():
        emergence_steps[family] = stable_threshold_step(
            steps,
            values,
            threshold=threshold,
            min_consecutive=min_consecutive,
        )

    non_control_steps = {
        family: step
        for family, step in emergence_steps.items()
        if family != control_name and step is not None
    }
    if non_control_steps:
        earliest_family = min(non_control_steps, key=lambda item: (non_control_steps[item], item))
        latest_family = max(non_control_steps, key=lambda item: (non_control_steps[item], item))
    else:
        earliest_family = None
        latest_family = None

    non_control_count = sum(family != control_name for family in family_values)
    random_control_passed = True
    if control_name in family_values:
        random_control_passed = random_control_report(
            family_values[control_name],
            max_allowed_value=threshold,
        ).control_passed

    return DevelopmentalComparisonReport(
        threshold=threshold,
        emergence_steps=emergence_steps,
        earliest_family=earliest_family,
        latest_family=latest_family,
        random_control_passed=random_control_passed,
        all_non_control_emerged=len(non_control_steps) == non_control_count,
    )


tests.test_developmental_comparison_excludes_random_control_from_ordering(
    developmental_comparison_report
)


All tests in `test_developmental_comparison_excludes_random_control_from_ordering` passed!


## Exercise 5 - Live Checkpoint Archaeology

Toy trajectories teach the report logic, but checkpoint archaeology is about saved training states. This exercise trains a tiny mod-13 addition model, saves checkpoints, reloads them before measuring, and compares the target run with a random-label control on the same schedule.

> Difficulty: medium-hard  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_live_checkpoint_archaeology_smoke_test_trains_saves_reloads_and_controls` passed!
```

</details>

<details>
<summary>Help - why reload the checkpoint files?</summary>

Reloading proves the trajectory can be reconstructed from historical artifacts. Measuring only the current in-memory model would not be checkpoint archaeology.

</details>

<details>
<summary>Common bugs</summary>

- Saving only the final checkpoint.
- Measuring before reloading.
- Training the random-label control on a different checkpoint schedule.
- Describing complete finite-domain evaluation as OOD generalization.

</details>

<details>
<summary>Solution</summary>

Save every declared step, reload each file with `utils.checkpoint_metrics_from_file`, analyze the reloaded target trajectory, and require the random-label control to stay near chance.

</details>


In [6]:
def _train_save_reload_modular_run(
    checkpoint_dir: Path,
    *,
    device: t.device,
    seed: int,
    random_labels: bool = False,
) -> dict:
    t.manual_seed(seed)
    input_pairs, true_labels = utils.modular_addition_table(device=device)
    generator = t.Generator(device=device).manual_seed(seed + 1000)
    train_labels = true_labels
    if random_labels:
        train_labels = t.randint(
            0,
            utils.LIVE_MODULAR_ARCHAEOLOGY_MODULUS,
            true_labels.shape,
            generator=generator,
            device=device,
        )

    model = utils.TinyModularAdditionMLP().to(device)
    optimizer = t.optim.AdamW(
        model.parameters(),
        lr=utils.LIVE_MODULAR_ARCHAEOLOGY_LR,
        weight_decay=1e-3,
    )

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_paths: list[Path] = []
    checkpoint_steps = set(utils.LIVE_MODULAR_ARCHAEOLOGY_CHECKPOINT_STEPS)
    for step in range(utils.LIVE_MODULAR_ARCHAEOLOGY_STEPS + 1):
        if step in checkpoint_steps:
            path = checkpoint_dir / f"step_{step:04d}.pt"
            t.save(model.state_dict(), path)
            checkpoint_paths.append(path)
        if step == utils.LIVE_MODULAR_ARCHAEOLOGY_STEPS:
            break
        optimizer.zero_grad(set_to_none=True)
        loss = t.nn.functional.cross_entropy(model(input_pairs), train_labels)
        loss.backward()
        optimizer.step()

    reloaded_accuracies: list[float] = []
    reloaded_losses: list[float] = []
    for path in checkpoint_paths:
        accuracy, loss = utils.checkpoint_metrics_from_file(
            path,
            device=device,
            input_pairs=input_pairs,
            true_labels=true_labels,
        )
        reloaded_accuracies.append(accuracy)
        reloaded_losses.append(loss)

    return {
        "steps": t.tensor(utils.LIVE_MODULAR_ARCHAEOLOGY_CHECKPOINT_STEPS, device=device),
        "accuracies": t.tensor(reloaded_accuracies, device=device),
        "losses": t.tensor(reloaded_losses, device=device),
        "checkpoint_count": len(checkpoint_paths),
        "checkpoint_total_bytes": sum(path.stat().st_size for path in checkpoint_paths),
    }


def live_checkpoint_archaeology_smoke_test(
    checkpoint_root: Path | None = None,
    *,
    device: str | t.device = "cpu",
    seed: int = 0,
) -> dict:
    device = t.device(device)

    def _run(root: Path) -> dict:
        target = _train_save_reload_modular_run(
            root / "target_labels",
            device=device,
            seed=seed,
            random_labels=False,
        )
        random_control = _train_save_reload_modular_run(
            root / "random_labels",
            device=device,
            seed=seed,
            random_labels=True,
        )
        checkpoint_files = sorted(root.glob("*/*.pt"))
        target_emergence = mechanism_emergence_report(
            target["steps"],
            target["accuracies"],
            metric_name="modular_addition_table_accuracy",
            threshold=utils.LIVE_MODULAR_ARCHAEOLOGY_THRESHOLD,
            min_consecutive=utils.LIVE_MODULAR_ARCHAEOLOGY_MIN_CONSECUTIVE,
        )
        target_phase = phase_transition_report(
            target["steps"],
            target["accuracies"],
            metric_name="modular_addition_table_accuracy",
            min_jump=0.2,
        )
        random_report = random_control_report(
            random_control["accuracies"],
            metric_name="random_label_true_table_accuracy",
            max_allowed_value=0.2,
        )
        checkpoint_count = target["checkpoint_count"] + random_control["checkpoint_count"]
        final_accuracy = float(target["accuracies"][-1].item())
        random_control_peak = float(random_control["accuracies"].max().item())

        return {
            "preflight_passed": (
                target_emergence.emerged
                and final_accuracy >= 0.99
                and target_phase.phase_transition_detected
                and random_report.control_passed
                and random_control_peak <= 0.2
                and len(checkpoint_files) == checkpoint_count
            ),
            "device": str(device),
            "model_family": "tiny_modular_addition_mlp",
            "modulus": utils.LIVE_MODULAR_ARCHAEOLOGY_MODULUS,
            "table_example_count": utils.LIVE_MODULAR_ARCHAEOLOGY_MODULUS**2,
            "complete_finite_domain_evaluated": True,
            "ood_generalization_claimed": False,
            "generalization_scope": (
                "Complete finite mod-13 addition table (169/169 input pairs); "
                "no held-out OOD extrapolation is claimed."
            ),
            "checkpoint_steps": utils.LIVE_MODULAR_ARCHAEOLOGY_CHECKPOINT_STEPS,
            "checkpoint_count": checkpoint_count,
            "checkpoint_files_written": len(checkpoint_files),
            "checkpoint_total_bytes": (
                target["checkpoint_total_bytes"] + random_control["checkpoint_total_bytes"]
            ),
            "target_accuracy_trajectory": target["accuracies"].detach().cpu().tolist(),
            "target_loss_trajectory": target["losses"].detach().cpu().tolist(),
            "random_control_accuracy_trajectory": random_control["accuracies"].detach().cpu().tolist(),
            "first_crossing_step": target_emergence.first_crossing_step,
            "stable_from_step": target_emergence.stable_from_step,
            "final_accuracy": final_accuracy,
            "phase_transition_step": target_phase.transition_step,
            "phase_transition_jump": target_phase.jump,
            "phase_transition_detected": target_phase.phase_transition_detected,
            "random_control_peak_accuracy": random_report.peak_value,
            "random_control_passed": random_report.control_passed,
            "real_checkpoints_reloaded": True,
        }

    if checkpoint_root is not None:
        root = Path(checkpoint_root)
        root.mkdir(parents=True, exist_ok=True)
        return _run(root)

    with tempfile.TemporaryDirectory(prefix="arena17_live_checkpoint_archaeology_") as tmp:
        return _run(Path(tmp))


tests.test_live_checkpoint_archaeology_smoke_test_trains_saves_reloads_and_controls(
    live_checkpoint_archaeology_smoke_test
)


All tests in `test_live_checkpoint_archaeology_smoke_test_trains_saves_reloads_and_controls` passed!


## Exercise 6 - Notebook Contract and CUDA Report

The notebook contract is the CPU-feasible path students run while working. The committed CUDA report is produced by the section verification runner and records the same finite-table checkpoint archaeology on GPU.

> Difficulty: medium  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_checkpoint_emergence_smoke_test` passed!
All tests in `test_phase_transition_smoke_test` passed!
All tests in `test_random_control_smoke_test` passed!
All tests in `test_developmental_comparison_smoke_test` passed!
All tests in `test_notebook_contract` passed!
All tests in `test_committed_gpu_report_records_real_checkpoint_preflight` passed!
```

</details>

<details>
<summary>Help - why read the committed CUDA report?</summary>

The notebook should stay fast and inspectable. The full acceptance path is regenerated by `scripts/run_extension_verification_reports.py --section 17.1`, then the notebook reads the committed JSON report and checks the same scoped evidence.

</details>

<details>
<summary>Common bugs</summary>

- Returning dataclass objects instead of JSON-serializable dicts.
- Dropping the live checkpoint path from `run_smoke_test`.
- Treating the CPU smoke path as a substitute for the CUDA report.

</details>

<details>
<summary>Solution</summary>

Expose small smoke-test dictionaries, read `verification_report.json` for the committed GPU metrics, and keep finite-domain scope explicit.

</details>


In [7]:
def _report_dict(report) -> dict:
    return report.__dict__.copy()


def checkpoint_emergence_smoke_test() -> dict:
    steps, trajectories = toy_training_trajectories()
    return _report_dict(
        mechanism_emergence_report(
            steps,
            trajectories["autoregressive"],
            metric_name="induction_probe_accuracy",
            threshold=0.6,
            min_consecutive=2,
        )
    )


def phase_transition_smoke_test() -> dict:
    steps, trajectories = toy_training_trajectories()
    return _report_dict(
        phase_transition_report(
            steps,
            trajectories["autoregressive"],
            metric_name="induction_probe_accuracy",
            min_jump=0.3,
        )
    )


def random_control_smoke_test() -> dict:
    _, trajectories = toy_training_trajectories()
    return _report_dict(
        random_control_report(
            trajectories["random_control"],
            metric_name="label_shuffled_probe",
            max_allowed_value=0.2,
        )
    )


def developmental_comparison_smoke_test() -> dict:
    steps, trajectories = toy_training_trajectories()
    return _report_dict(
        developmental_comparison_report(
            steps,
            trajectories,
            threshold=0.6,
            min_consecutive=2,
        )
    )


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "checkpoint_emergence": checkpoint_emergence_smoke_test(),
        "phase_transition": phase_transition_smoke_test(),
        "random_control": random_control_smoke_test(),
        "developmental_comparison": developmental_comparison_smoke_test(),
        "live_checkpoint_archaeology": live_checkpoint_archaeology_smoke_test(device="cpu"),
    }


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    tests.test_committed_gpu_report_records_real_checkpoint_preflight()
    report = json.loads((section_dir / "verification_report.json").read_text())
    metrics = report["metrics"]["gpu_test"]
    if metrics["peak_vram_gb"] > max_vram_gb:
        raise AssertionError(f"report used {metrics['peak_vram_gb']:.3f} GB VRAM")
    return metrics


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


tests.test_checkpoint_emergence_smoke_test(checkpoint_emergence_smoke_test)
tests.test_phase_transition_smoke_test(phase_transition_smoke_test)
tests.test_random_control_smoke_test(random_control_smoke_test)
tests.test_developmental_comparison_smoke_test(developmental_comparison_smoke_test)
tests.test_notebook_contract(run_smoke_test)
report = run_gpu_test()
{
    "device": report["device"],
    "final_accuracy": report["final_accuracy"],
    "stable_from_step": report["stable_from_step"],
    "phase_transition_jump": report["phase_transition_jump"],
    "random_control_peak_accuracy": report["random_control_peak_accuracy"],
    "peak_vram_gb": report["peak_vram_gb"],
}


All tests in `test_checkpoint_emergence_smoke_test` passed!
All tests in `test_phase_transition_smoke_test` passed!
All tests in `test_random_control_smoke_test` passed!
All tests in `test_developmental_comparison_smoke_test` passed!


All tests in `test_notebook_contract` passed!
All tests in `test_committed_gpu_report_records_real_checkpoint_preflight` passed!


{'device': 'NVIDIA GeForce RTX 5090 Laptop GPU',
 'final_accuracy': 1.0,
 'stable_from_step': 30,
 'phase_transition_jump': 0.2781065106391907,
 'random_control_peak_accuracy': 0.10059171915054321,
 'peak_vram_gb': 0.06275796890258789}

## Signature Result

A convincing 17.1 result is not a single training curve. It is a bounded checkpoint report where the target run, saved files, reload metrics, stable threshold, phase jump, and random-label control all agree.

| Check | Passing result |
|---|---:|
| Complete finite table evaluated | `169 / 169` mod-13 pairs |
| Real checkpoints written and reloaded | `26` |
| First threshold crossing | step `30` |
| Stable threshold crossing | step `30` |
| Final target accuracy | `1.0` |
| Largest adjacent phase jump | `0.2781` |
| Random-label peak true-table accuracy | `0.1006` |
| Peak VRAM | `0.063 GB` |

<details>
<summary>Interpreting the result</summary>

This supports the scoped claim: for a generated finite mod-13 model organism, saved and reloaded checkpoints show stable table-accuracy emergence at step 30, a real adjacent jump, and a random-label control that does not look emergent.

</details>

## Limitations

Supported: deterministic toy trajectories, stable-threshold and phase-transition reports, random-label controls, CPU-feasible live checkpoint archaeology, and CUDA finite-table mod-13 save/reload evidence.

Not supported: large-model checkpoint archaeology, real transformer mechanism discovery, OOD generalization, causal localization inside the MLP, or real AR/JEPA/diffusion/Mamba training comparisons.

Deferred: real transformer checkpoint archaeology, activation-level mechanism localization, grokking curve replication, and checkpoint-difference circuit tracing.
